# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shakir-j/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I chose Logistic Regression because the lane is a binary classification task: predict whether content will show March activity using signals that were available in February. Logistic Regression is simple, interpretable, and provides a useful probability-based baseline for comparing feature signals without adding unnecessary model complexity. The model will use only February-derived features and will be evaluated on the same split and metric used for the Week-4 baseline.

In [12]:
# ============================================================
# WEEK 5 — DATA SETUP
# Reuse the verified FlyRank warehouse connection from Week 4
# ============================================================

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

# ------------------------------------------------------------
# Hugging Face authentication
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise RuntimeError(
        "HF_TOKEN not found. Check Colab Secrets and make sure "
        "the secret is named exactly HF_TOKEN."
    )

# ------------------------------------------------------------
# DuckDB connection
# ------------------------------------------------------------

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

# ------------------------------------------------------------
# Same verified warehouse paths used in Week 4
# ------------------------------------------------------------

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Outcome window: March 2026")


# ============================================================
# BUILD MODEL DATASET
# ============================================================

feature_frame = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(COALESCE(gsc_impressions, 0))
            AS feb_gsc_impressions,

        SUM(COALESCE(gsc_clicks, 0))
            AS feb_gsc_clicks,

        AVG(gsc_avg_position)
            AS feb_gsc_avg_position,

        SUM(COALESCE(ga4_sessions, 0))
            AS feb_ga4_sessions,

        SUM(COALESCE(scroll_events, 0))
            AS feb_scroll_events

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        CASE
            WHEN
                SUM(COALESCE(gsc_clicks, 0)) > 0
                OR SUM(COALESCE(ga4_sessions, 0)) > 0
                OR SUM(COALESCE(scroll_events, 0)) > 0
            THEN 1
            ELSE 0
        END AS march_activity_label

    FROM {MAR}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,

    f.feb_gsc_impressions,
    f.feb_gsc_clicks,
    f.feb_gsc_avg_position,
    f.feb_ga4_sessions,
    f.feb_scroll_events,

    COALESCE(m.march_activity_label, 0)
        AS march_activity_label

FROM feb f

LEFT JOIN mar m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id
""").df()


# ------------------------------------------------------------
# Model features and target
# ------------------------------------------------------------

feature_cols = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_ga4_sessions",
    "feb_scroll_events"
]

X = feature_frame[feature_cols].copy()
y = feature_frame["march_activity_label"].astype(int)


# ------------------------------------------------------------
# Basic verification
# ------------------------------------------------------------

print("\nFeature frame shape:", feature_frame.shape)
print("Feature columns:", feature_cols)
print("Positive label rate:", round(y.mean(), 4))
print("Missing values:")
print(X.isna().sum())

Connected to FlyRank warehouse.
Feature window: February 2026
Outcome window: March 2026

Feature frame shape: (321546, 8)
Feature columns: ['feb_gsc_impressions', 'feb_gsc_clicks', 'feb_gsc_avg_position', 'feb_ga4_sessions', 'feb_scroll_events']
Positive label rate: 0.3176
Missing values:
feb_gsc_impressions          0
feb_gsc_clicks               0
feb_gsc_avg_position    167987
feb_ga4_sessions             0
feb_scroll_events            0
dtype: int64


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression

method_name = "Logistic Regression"

print("Method:", method_name)
print("Task: Binary classification")
print("Reason: interpretable, simple, and suitable for predicting March activity from February signals.")

Method: Logistic Regression
Task: Binary classification
Reason: interpretable, simple, and suitable for predicting March activity from February signals.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I use a client-grouped 80/20 train-test split so that the same client does not appear in both sets. This gives a more honest test of whether the model can generalize to unseen clients. The model uses only February-derived features, while the March activity label is used only as the target.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 2 — SPLIT DESIGN
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

# Use client as the grouping unit
groups = feature_frame["client_hash_id"]

# 80/20 client-grouped split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

# Create train/test sets
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Verify client separation
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

client_overlap = train_clients.intersection(test_clients)

print("Split design: client-grouped 80/20 split")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))
print("Training positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))


Split design: client-grouped 80/20 split
Training rows: 268596
Test rows: 52950
Training clients: 43
Test clients: 11
Client overlap: 0
Training positive rate: 0.3071
Test positive rate: 0.3708


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model and baseline comparison

I use Logistic Regression because the target is binary and the model provides a simple, interpretable benchmark. Missing numeric values are handled inside the pipeline with median imputation.

I compare the model with the Week-4 rule on the same client-grouped test set and use accuracy, precision, recall, and F1 so that the comparison is not based on one metric alone.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 3 — TRAIN + COMPARE VS WEEK-4 BASELINE
# ============================================================

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ------------------------------------------------------------
# Logistic Regression model
# ------------------------------------------------------------

model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000, random_state=42)
)

model.fit(X_train, y_train)

# Model predictions
model_predictions = model.predict(X_test)

# ------------------------------------------------------------
# Week-4 baseline rule
#
# Meaningful visibility:
#   impressions >= 100
#
# Top-20 position:
#   average position <= 20
#
# Weak CTR:
#   CTR < 3%
# ------------------------------------------------------------

test_features = X_test.copy()

test_features["ctr_pct"] = (
    100.0
    * test_features["feb_gsc_clicks"]
    / test_features["feb_gsc_impressions"].replace(0, float("nan"))
)

baseline_predictions = (
    (test_features["feb_gsc_impressions"] >= 100)
    & (test_features["feb_gsc_avg_position"] <= 20)
    & (test_features["ctr_pct"] < 3)
).astype(int)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

results = pd.DataFrame({
    "Model": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(y_test, baseline_predictions),
        accuracy_score(y_test, model_predictions)
    ],
    "Precision": [
        precision_score(y_test, baseline_predictions, zero_division=0),
        precision_score(y_test, model_predictions, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_predictions, zero_division=0),
        recall_score(y_test, model_predictions, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, baseline_predictions, zero_division=0),
        f1_score(y_test, model_predictions, zero_division=0)
    ]
})

print("MODEL VS BASELINE")
print(results.round(4).to_string(index=False))

print("\nTest rows:", len(X_test))
print("Model positive predictions:", int(model_predictions.sum()))
print("Baseline positive predictions:", int(baseline_predictions.sum()))


MODEL VS BASELINE
              Model  Accuracy  Precision  Recall     F1
    Week-4 baseline    0.8044     0.8098  0.6176 0.7008
Logistic Regression    0.7837     0.8896  0.4758 0.6200

Test rows: 52950
Model positive predictions: 10502
Baseline positive predictions: 14976


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The Logistic Regression model has higher precision than the Week-4 baseline, but lower recall and F1, and its accuracy is also lower. This means the model is more selective when predicting March activity, but it misses more positive cases. The baseline therefore remains stronger on this test split overall.

The model is influenced by the February search and engagement signals, but these signals do not capture all of the variation in March activity. The errors suggest that February behavior alone is not sufficient to identify every future active item.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 4 — ERRORS AND INTERPRETATION
# ============================================================

from sklearn.metrics import confusion_matrix

# ------------------------------------------------------------
# Confusion matrix for Logistic Regression
# ------------------------------------------------------------

tn, fp, fn, tp = confusion_matrix(
    y_test,
    model_predictions
).ravel()

print("LOGISTIC REGRESSION ERROR ANALYSIS")
print("----------------------------------")
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

false_positive_rate = fp / max(fp + tn, 1)
false_negative_rate = fn / max(fn + tp, 1)

print("\nFalse positive rate:", round(false_positive_rate, 4))
print("False negative rate:", round(false_negative_rate, 4))

# ------------------------------------------------------------
# Feature interpretation
# ------------------------------------------------------------

model_lr = model.named_steps["logisticregression"]

coefficients = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model_lr.coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("\nFEATURE COEFFICIENTS")
print(coefficients[
    ["feature", "coefficient"]
].to_string(index=False))

print("\nINTERPRETATION")
print(
    "The model produces more false negatives than false positives, "
    "which is consistent with its higher precision but lower recall. "
    "The largest coefficient magnitudes indicate which February "
    "features have the strongest directional influence on predictions."
)


LOGISTIC REGRESSION ERROR ANALYSIS
----------------------------------
True negatives: 32155
False positives: 1159
False negatives: 10293
True positives: 9343

False positive rate: 0.0348
False negative rate: 0.5242

FEATURE COEFFICIENTS
             feature  coefficient
    feb_ga4_sessions     1.024362
      feb_gsc_clicks     0.423213
   feb_scroll_events     0.322377
feb_gsc_avg_position    -0.006745
 feb_gsc_impressions     0.001669

INTERPRETATION
The model produces more false negatives than false positives, which is consistent with its higher precision but lower recall. The largest coefficient magnitudes indicate which February features have the strongest directional influence on predictions.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.